# Notebook 01 — JIT Data Scraping

**Student:** Rohit Sharma | 19385207

This notebook collects a just-in-time (JIT) dataset of live Google Play Store reviews using the `google-play-scraper` library.

---

## Step 1: Install dependencies

In [13]:
# Run this cell first (only needed once in Colab)
!pip install google-play-scraper pandas

zsh:1: command not found: pip


## Step 2: Import libraries

In [15]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time
import os

print('Libraries imported successfully')

Libraries imported successfully


## Step 3: Define apps to scrape

We scrape from a variety of app categories to ensure diversity in our dataset.

In [16]:
# App IDs from Google Play Store — covering different categories
APPS = {
    'productivity': [
        'com.todoist',           # Todoist
        'com.microsoft.teams',   # Microsoft Teams
        'com.microsoft.todos',         # Microsoft To Do
    ],
    'entertainment': [
        'com.netflix.mediaclient',  # Netflix
        'com.spotify.music',        # Spotify
    ],
    'health': [
        'com.myfitnesspal.android',  # MyFitnessPal
        'com.calm.android',     # Calm
    ],
    'social': [
        'com.instagram.android',  # Instagram
        'org.telegram.messenger',           # Telegram
    ],
    'finance': [
        'com.paypal.android.p2pmobile',  # PayPal
        'com.revolut.revolut',             # Revolut
    ]
}

print(f'Total apps to scrape: {sum(len(v) for v in APPS.values())}')

Total apps to scrape: 11


## Step 4: Scrape reviews

In [17]:
def scrape_app_reviews(app_id, category, count=200):
    """
    Scrape reviews for a single app.
    Returns a list of review dictionaries.
    """
    try:
        result, _ = reviews(
            app_id,
            lang='en',
            country='us',
            sort=Sort.NEWEST,
            count=count
        )
        for r in result:
            r['app_id'] = app_id
            r['category'] = category
        print(f'  Scraped {len(result)} reviews for {app_id}')
        return result
    except Exception as e:
        print(f'  ERROR scraping {app_id}: {e}')
        return []


all_reviews = []

for category, app_list in APPS.items():
    print(f'\nScraping category: {category}')
    for app_id in app_list:
        reviews_data = scrape_app_reviews(app_id, category, count=300)
        all_reviews.extend(reviews_data)
        time.sleep(2)  # Be polite — avoid rate limiting

print(f'\nTotal reviews collected: {len(all_reviews)}')


Scraping category: productivity
  Scraped 300 reviews for com.todoist
  Scraped 300 reviews for com.microsoft.teams
  Scraped 300 reviews for com.microsoft.todos

Scraping category: entertainment
  Scraped 300 reviews for com.netflix.mediaclient
  Scraped 300 reviews for com.spotify.music

Scraping category: health
  Scraped 300 reviews for com.myfitnesspal.android
  Scraped 300 reviews for com.calm.android

Scraping category: social
  Scraped 300 reviews for com.instagram.android
  Scraped 300 reviews for org.telegram.messenger

Scraping category: finance
  Scraped 300 reviews for com.paypal.android.p2pmobile
  Scraped 300 reviews for com.revolut.revolut

Total reviews collected: 3300


## Step 5: Convert to DataFrame and inspect

In [18]:
df = pd.DataFrame(all_reviews)

# Keep only the columns we need
df = df[['reviewId', 'content', 'score', 'at', 'app_id', 'category']]

# Rename for clarity
df.columns = ['review_id', 'review_text', 'star_rating', 'date', 'app_id', 'app_category']

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nStar rating distribution:')
print(df['star_rating'].value_counts().sort_index())
df.head(10)

Dataset shape: (3300, 6)

Columns: ['review_id', 'review_text', 'star_rating', 'date', 'app_id', 'app_category']

Star rating distribution:
star_rating
1     955
2     204
3     212
4     262
5    1667
Name: count, dtype: int64


,review_id,review_text,star_rating,date,app_id,app_category
0,92198fd0-aab5-40c5-a71b-7c7c462400af,awesome helps me get things done without forge...,5,2026-07-08 18:30:42,com.todoist,productivity
1,28364916-8d16-4293-b592-57733c7ffcb3,a helpful tool with the right process,5,2026-07-08 07:00:19,com.todoist,productivity
2,6f18edda-b654-4fb7-a618-957ed2590efa,"It was useless during Iran Internet shutdown, ...",1,2026-07-08 03:51:44,com.todoist,productivity
3,130a3742-626a-4133-8c39-afea0d8e5e49,wouls be nice to have more control over settin...,3,2026-07-08 01:59:32,com.todoist,productivity
4,d3df7fe1-5686-445e-8122-39dd81e2bfc7,I had an issue and appreciated the quick respo...,5,2026-07-07 08:43:07,com.todoist,productivity
5,7b6b8a00-cd2a-4196-bc25-8ddd02897ed9,very good for management of tasks and well mai...,5,2026-07-07 01:40:06,com.todoist,productivity
6,14561524-876a-49cd-94a0-e089b45242b3,focus detail after use it.,4,2026-07-06 16:22:02,com.todoist,productivity
7,bbee8457-d574-4921-87a5-60e0cc78fc41,I am a busy guy and I do not see why I am forc...,3,2026-07-06 16:12:25,com.todoist,productivity
8,33fad263-1715-4da0-a4e8-63904f0e780b,i dont like this app and i cant unsuscribe to ...,1,2026-07-06 06:32:42,com.todoist,productivity
9,49818736-0873-4ce3-976a-ac7da9044058,❤️,5,2026-07-06 05:31:05,com.todoist,productivity


## Step 6: Basic cleaning

In [19]:
# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset='review_text')
print(f'Removed {before - len(df)} duplicate reviews')

# Remove empty reviews
df = df.dropna(subset=['review_text'])
df = df[df['review_text'].str.strip() != '']

# Remove very short reviews (less than 5 words) — not useful for classification
df = df[df['review_text'].str.split().str.len() >= 5]

print(f'Final dataset size: {len(df)} reviews')
df.head()

Removed 318 duplicate reviews
Final dataset size: 2170 reviews


,review_id,review_text,star_rating,date,app_id,app_category
0,92198fd0-aab5-40c5-a71b-7c7c462400af,awesome helps me get things done without forge...,5,2026-07-08 18:30:42,com.todoist,productivity
1,28364916-8d16-4293-b592-57733c7ffcb3,a helpful tool with the right process,5,2026-07-08 07:00:19,com.todoist,productivity
2,6f18edda-b654-4fb7-a618-957ed2590efa,"It was useless during Iran Internet shutdown, ...",1,2026-07-08 03:51:44,com.todoist,productivity
3,130a3742-626a-4133-8c39-afea0d8e5e49,wouls be nice to have more control over settin...,3,2026-07-08 01:59:32,com.todoist,productivity
4,d3df7fe1-5686-445e-8122-39dd81e2bfc7,I had an issue and appreciated the quick respo...,5,2026-07-07 08:43:07,com.todoist,productivity


## Step 7: Save raw dataset

In [20]:
# Save to CSV
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/reviews_raw.csv', index=False)
print(f'Saved {len(df)} reviews to ../data/raw/reviews_raw.csv')


Saved 2170 reviews to ../data/raw/reviews_raw.csv


---
## Summary

| Item | Value |
|------|-------|
| Total reviews collected | See output above |
| Apps scraped | 10 apps across 5 categories |
| Language | English only |
| Output file | `data/raw/reviews_raw.csv` |
